In [ ]:
#Set working directory to the location of your clustering code folder
import os
os.chdir('')

In [ ]:
#Imports the functions from other scripts
#If you make any changes to the underlying scripts you will need to rerun the code from here for those changes to apply
import yaml
import os

import sys
sys.path.insert(1, "../")

from src.data_functions import *
from src.clustering_functions import *
from src.dissemination_functions import *
from yaml.loader import SafeLoader

In [ ]:
#Read in the config file
clustering_refactor_folder_path = os.path.abspath(os.path.join(os.path.realpath('__file__'), '../..'))
config_path = f"config.yaml".replace("\\", "/")
with open(config_path, encoding="utf-8") as f:
    loaded_config = yaml.load(f, Loader=SafeLoader)

In [ ]:
#Load data and split into individual metrics
#This will include all metrics specified in the config
datasets = import_data(
    loaded_config=loaded_config,
    cols_to_select=["AREACD", "AREANM","Indicator", "Value"],
    table_name=loaded_config["subnational_indicators_table_name"],
)

In [ ]:
#Cleans the data, including UTLA to LTLA imputation
for key, value in datasets.items():
    value = clean_groups(loaded_config, value)

In [ ]:
#Convert metrics into pivot tables, your specified data is now stored as tables['custom_metrics']
tables = {}
for key, value in datasets.items():
    tables[key] = metrics_to_table(value)

In [ ]:
#Set the max rows displayed as 500 if spot checking in script
pd.set_option('display.max_rows', 10)

In [ ]:
#Inspect the data to ensure all columns are present
tables['custom_metrics'].head()

In [ ]:
#Fuction subsets the data for the specified geography type using the names and codes file
#Names and codes file referred to as Geog_mapper in config
#This can be adapted through the config to run on any geography type or subset of geography
cluster_df = get_desired_geography(
    loaded_config= loaded_config,
    df= tables['custom_metrics'],
)
cluster_df

In [ ]:
#Removes any NAs if created by previous function
cluster_df = cluster_df.dropna(subset='AREACD')
cluster_df

In [ ]:
#This function takes the dataset and computes pearsons correlation between all metrics
correlation_matrix = get_correlation_matrix(df= cluster_df)
correlation_matrix

In [ ]:
#This function outputs a data frame of the winzorisation thresholds for each metric
#Currently set to 1st and 99th percentile, this can be altered
thresholds = get_winsorization_thresholds(
    df=cluster_df,
    lower_threshold = 0.01,
    upper_threshold = 0.99,
)
thresholds

In [ ]:
#This function Winsorizes the data
#Winsorization deals with extreme values by setting any value above/below a certain percentile to the value of that percentile
#Currently set to 1st and 99th percentile, this can be altered
cluster_df_win = winsorize(
    df=cluster_df,
    lower_threshold = 0.01,
    upper_threshold = 0.99,
)

In [ ]:
#This function takes the winsorized data (or you can use the base data cluster_df) and runs the kmeans model
#The data you wish to cluster should be specified in the metrics parameter
#n_init specifies the number of times the model is to be run, recommended 100 for initial  tests and 10000 for final output
#Setting min and max k specifies the range of the number of clusters to test, the code takes longer to run for wider ranges
#A geodataframe including clusters, the cluster centres (for radar plot) and a silouette score df are output
cluster_geodataframe, cluster_centres, sil_score = make_clustering_model(
    loaded_config=loaded_config,
    metrics=cluster_df_win,
    n_init=100,
    min_k=4,
    max_k=5,
)
cluster_geodataframe

In [ ]:
#This function alters the geodataframe to create a readable cluster table
cluster_table = cluster_table(
    loaded_config=loaded_config,
    clusters_table=cluster_geodataframe,
) 
cluster_table

In [ ]:
#This function uses the geodataframe to create a map showing the cluster of each area
#This map is automatically saved into the output folder and can be called into the excel output at the end of this script
cluster_map = cluster_map(
    loaded_config= loaded_config,
    clusters=cluster_geodataframe, 
)

In [ ]:
#This function uses the geodataframe, the cluster centres and the metric dataframe to create a radar plot
#This plot is automatically saved into the output folder and can be called into the excel output at the end of this script
#For models with more than 6 metrics variable names may overlap on the radar plot
radar_plot = radar_plot(
    loaded_config= loaded_config,
    metrics= cluster_df,
    clusters= cluster_geodataframe,
    centres = cluster_centres,
)

In [ ]:
#This function takes the cluster geodataframe and creates a table showing how the areas in each cluster are distributed across the UK regions
#This may not be necessary for some projects and can be removed
ITL1_table = ITL1_summary(
    loaded_config=loaded_config,
    clusters_table=cluster_geodataframe,
) 
ITL1_table

In [ ]:
#This function takes the cluster geodataframe to create a table of mean values for each metric by cluster
#The total column in this table shows an average of the values of the the desired geographies
#This is not the same as a UK average and may differ from other published estimates
mean_table = clusters_summary_stats(
    table_metrics= cluster_df,
    clusters_table= cluster_geodataframe,
    stats= "mean",
)
mean_table

In [ ]:
#This function takes the cluster geodataframe to create a table of median values for each metric by cluster
#The total column in this table shows an average of the values of the the desired geographies
#This is not the same as a UK average and may differ from other published estimates
median_table = clusters_summary_stats(
    table_metrics= cluster_df,
    clusters_table= cluster_geodataframe,
    stats= "median",
)
median_table

In [ ]:
##This section of the code conducts the variance and PCA analysis used for variable selection

In [ ]:
#This function conducts the PCA analysis with an important feature threshold of 0.25 (can be changed)
principal_components, loading_df, important_features = pca_analysis(
    loaded_config=loaded_config,
    df=cluster_df,
    threshold = 0.25,
)

In [ ]:
#This function creates a table of the variance of each metric included
variance_table = variance_analysis(
    loaded_config=loaded_config,
    df=cluster_df,
)

In [ ]:
#This function creates a histogram of pairwise distances
#This histogram is saved in the "output" folder as a jpeg named "Histogram of Pairwise Distances"
pairwise_distances = visualize_pairwise_distances(
    loaded_config=loaded_config,
    df=cluster_df,
)

In [ ]:
#This function exports all relevant data to a single xlsx file in the outputs folder
#Including the visualisations can be specified by the boolean operator include_maps
#If not all data is required, remove unnecessary objects from frames
#File name must be specified with the ".xlsx" file type not included or it won't work
export_to_xlsx(
    loaded_config=loaded_config,
    frames = {'Cluster_table': cluster_table, 'ITL1_table': ITL1_table, 'Silhoutte_score': sil_score, 
              'Cluster_medians': median_table, 'Cluster_means': mean_table,'correlation_matrix': correlation_matrix,
              'data': cluster_df,
              'principal_components': principal_components, 'loadings': loading_df,
              'variance_table': variance_table
             },
    file_name = "",
    include_maps = True,
)